In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [ ]:
# Load and combine
df_2023 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2023_c20260323.csv')
df_2024 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2024_c20260323 2.csv')
df = pd.concat([df_2023, df_2024], ignore_index=True)

# Filter to top 10 event types
top_10_types = df['EVENT_TYPE'].value_counts().head(10).index.tolist()
print(df['EVENT_TYPE'].value_counts().head(10))
df = df[df['EVENT_TYPE'].isin(top_10_types)].copy()

# Feature engineering (all vectorized, no loops)
month_map = {'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
             'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}
df['MONTH'] = df['MONTH_NAME'].map(month_map)

# Hour from BEGIN_TIME (stored as HHMM integer, e.g. 1430 -> 14)
df['HOUR'] = df['BEGIN_TIME'] // 100

# Duration in minutes from full timestamps
df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['DURATION_MIN'] = (df['END_DT'] - df['BEGIN_DT']).dt.total_seconds() / 60

# CZ_TYPE to binary
df['CZ_TYPE_ENC'] = (df['CZ_TYPE'] == 'C').astype(int)

# Label encode state
le_state = LabelEncoder()
df['STATE_ENC'] = le_state.fit_transform(df['STATE'].astype(str))

# Label encode WFO
le_wfo = LabelEncoder()
df['WFO_ENC'] = le_wfo.fit_transform(df['WFO'].astype(str))

# Encode target labels (XGBoost wants integer classes)
le_y = LabelEncoder()
y_enc = le_y.fit_transform(df['EVENT_TYPE'])

# Stratified split (before imputation to prevent data leakage)
feature_cols = ['MONTH', 'HOUR', 'STATE_ENC', 'BEGIN_LAT', 'BEGIN_LON',
                'DURATION_MIN', 'CZ_TYPE_ENC', 'WFO_ENC']
X = df[feature_cols]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, stratify=y_enc, test_size=0.2, random_state=42
)

# --- Hierarchical lat/lon imputation (fit on training data only) ---
# Compute median coords at three geographic levels from training rows that have coords
train_df = df.loc[X_train.index]
has_coords = train_df[train_df['BEGIN_LAT'].notna()]
median_cz = has_coords.groupby(['STATE', 'CZ_FIPS'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_wfo = has_coords.groupby(['STATE', 'WFO'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_state = has_coords.groupby('STATE')[['BEGIN_LAT', 'BEGIN_LON']].median()

# Apply hierarchical fill to full df, then rebuild X
df_imp = df.copy()
missing = df_imp['BEGIN_LAT'].isna()

# Level 1: (STATE, CZ_FIPS) — county/zone median
for (state, cz_fips), row in median_cz.iterrows():
    mask = missing & (df_imp['STATE'] == state) & (df_imp['CZ_FIPS'] == cz_fips)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

missing = df_imp['BEGIN_LAT'].isna()

# Level 2: (STATE, WFO) — forecast office region median
for (state, wfo), row in median_wfo.iterrows():
    mask = missing & (df_imp['STATE'] == state) & (df_imp['WFO'] == wfo)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

missing = df_imp['BEGIN_LAT'].isna()

# Level 3: STATE — state median (last resort)
for state, row in median_state.iterrows():
    mask = missing & (df_imp['STATE'] == state)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

# Rebuild X from imputed data using the same indices
X_imp = df_imp[feature_cols]
X_train = X_imp.loc[X_train.index]
X_test = X_imp.loc[X_test.index]

# Sanity checks
print(f"\nShape: {X_train.shape}")
print(f"Missing per column (train):\n{X_train.isna().sum()}")
print(f"Missing per column (test):\n{X_test.isna().sum()}")

In [ ]:
# from sklearn.model_selection import GridSearchCV, StratifiedKFold
#
# # Hyperparameter grid (proposal: learning rate, max depth, n_estimators, subsample)
# param_grid = {
#     'learning_rate': [0.01, 0.05, 0.1, 0.3],
#     'max_depth': [3, 5, 7, 9],
#     'n_estimators': [100, 300, 500, 700],
#     'subsample': [0.7, 0.8, 1.0],
# }
#
# base_model = xgb.XGBClassifier(
#     objective='multi:softmax',
#     num_class=10,
#     random_state=42,
#     tree_method='hist',
# )
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#
# grid_search = GridSearchCV(
#     estimator=base_model,
#     param_grid=param_grid,
#     scoring='f1_macro',
#     cv=cv,
#     verbose=1,
#     n_jobs=-1,
# )
#
# grid_search.fit(X_train, y_train)
#
# print(f"\nBest CV Macro F1: {grid_search.best_score_:.4f}")
# print(f"Best params: {grid_search.best_params_}")
#
# # Use the best model for evaluation
# #model = grid_search.best_estimator_

# Best hyperparameters from the Week 2 grid search (5-fold stratified CV, 192 candidates).
# Skipping re-run — those results are already logged in the Week 2 report.
best_params = {
    'learning_rate': 0.1,
    'max_depth': 9,
    'n_estimators': 500,
    'subsample': 0.7,
}
best_cv_macro_f1 = 0.8949  # from the Week 2 grid search

print(f"Using tuned XGBoost params: {best_params}")
print(f"Prior CV Macro F1: {best_cv_macro_f1:.4f}")

# Train the tuned model with the best params (used by the rest of the notebook)
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist',
    **best_params,
)
model.fit(X_train, y_train)

In [ ]:
# Train default XGBoost for comparison
model_default = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist',
)
model_default.fit(X_train, y_train)
y_pred_default = model_default.predict(X_test)
macro_f1_default = f1_score(y_test, y_pred_default, average='macro')

# Evaluate tuned model on test set
y_pred_tuned = model.predict(X_test)
macro_f1_tuned = f1_score(y_test, y_pred_tuned, average='macro')

print("=" * 70)
print("Default XGBoost")
print("=" * 70)
print(f"Macro F1: {macro_f1_default:.4f}\n")
print(classification_report(y_test, y_pred_default, target_names=le_y.classes_))

print("\n" + "=" * 70)
print(f"Tuned XGBoost {best_params}")
print("=" * 70)
print(f"Macro F1: {macro_f1_tuned:.4f}\n")
print(classification_report(y_test, y_pred_tuned, target_names=le_y.classes_))

print("\n" + "=" * 70)
print(f"Improvement: {macro_f1_default:.4f} -> {macro_f1_tuned:.4f} ({macro_f1_tuned - macro_f1_default:+.4f})")
print("=" * 70)

In [ ]:
# Confusion matrices side by side
cm_default = confusion_matrix(y_test, y_pred_default)
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

print("=" * 70)
print("Confusion Matrix — Default XGBoost")
print("=" * 70)
print(pd.DataFrame(cm_default, index=le_y.classes_, columns=le_y.classes_))

print("\n" + "=" * 70)
print("Confusion Matrix — Tuned XGBoost")
print("=" * 70)
print(pd.DataFrame(cm_tuned, index=le_y.classes_, columns=le_y.classes_))

# Highlight Hail <-> Thunderstorm Wind confusion (dominant error pattern)
print("\n" + "=" * 70)
print("Hail <-> Thunderstorm Wind confusion")
print("=" * 70)
hail_idx = list(le_y.classes_).index('Hail')
tsw_idx = list(le_y.classes_).index('Thunderstorm Wind')
print(f"  Default — Hail predicted as TStorm Wind: {cm_default[hail_idx, tsw_idx]}")
print(f"  Tuned   — Hail predicted as TStorm Wind: {cm_tuned[hail_idx, tsw_idx]}")
print(f"  Default — TStorm Wind predicted as Hail: {cm_default[tsw_idx, hail_idx]}")
print(f"  Tuned   — TStorm Wind predicted as Hail: {cm_tuned[tsw_idx, hail_idx]}")

# Feature importance comparison
imp_default = pd.Series(model_default.feature_importances_, index=feature_cols)
imp_tuned = pd.Series(model.feature_importances_, index=feature_cols)
imp_cmp = pd.DataFrame({
    'Default': imp_default,
    'Tuned': imp_tuned,
    'Diff': imp_tuned - imp_default,
}).sort_values('Default', ascending=False)

print("\n" + "=" * 70)
print("Feature Importance Comparison")
print("=" * 70)
print(imp_cmp.to_string(float_format='%.4f'))

# TabNet Baseline

In [ ]:
import torch
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier

# Reuse the same X_train, X_test, y_train, y_test from the XGBoost cells above
# TabNet needs numpy float32 arrays
X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

# Categorical feature indices and dimensions for TabNet embeddings
cat_cols_tabnet = ['STATE_ENC', 'CZ_TYPE_ENC', 'WFO_ENC']
cat_idxs = [feature_cols.index(c) for c in cat_cols_tabnet]
cat_dims = [int(df_imp[c].nunique()) for c in cat_cols_tabnet]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tabnet = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1,
    n_d=8,
    n_a=8,
    n_steps=3,
    gamma=1.3,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device,
    verbose=10,
)

tabnet.fit(
    X_train_np, y_train,
    eval_set=[(X_test_np, y_test)],
    eval_metric=["accuracy"],
    max_epochs=100,
    patience=15,
    batch_size=1024,
)

In [ ]:
# TabNet evaluation
y_pred_tabnet = tabnet.predict(X_test_np)
macro_f1_tabnet = f1_score(y_test, y_pred_tabnet, average='macro')

print("=" * 70)
print("TabNet Baseline")
print("=" * 70)
print(f"Macro F1: {macro_f1_tabnet:.4f}\n")
print(classification_report(y_test, y_pred_tabnet, target_names=le_y.classes_))

# Compare all three models
print("\n" + "=" * 70)
print("Model Comparison (Macro F1)")
print("=" * 70)
print(f"  Default XGBoost:  {macro_f1_default:.4f}")
print(f"  Tuned XGBoost:    {macro_f1_tuned:.4f}")
print(f"  TabNet Baseline:  {macro_f1_tabnet:.4f}")

# Confusion matrix
cm_tabnet = confusion_matrix(y_test, y_pred_tabnet)
print("\n" + "=" * 70)
print("Confusion Matrix — TabNet")
print("=" * 70)
print(pd.DataFrame(cm_tabnet, index=le_y.classes_, columns=le_y.classes_))

# TabNet feature importances
imp_tabnet = pd.Series(tabnet.feature_importances_, index=feature_cols)
print("\n" + "=" * 70)
print("TabNet Feature Importance")
print("=" * 70)
print(imp_tabnet.sort_values(ascending=False).to_string(float_format='%.4f'))

# Ablation Study: CZ_TYPE Removed

Retrain all three models (default XGBoost, tuned XGBoost, TabNet) with `CZ_TYPE_ENC` dropped to measure its true contribution and verify the remaining meteorological features can carry performance.

In [ ]:
# Ablation: drop CZ_TYPE_ENC and retrain each model
feature_cols_ablation = [c for c in feature_cols if c != 'CZ_TYPE_ENC']
X_train_abl = X_train[feature_cols_ablation]
X_test_abl = X_test[feature_cols_ablation]

# Default XGBoost (no CZ_TYPE)
model_default_abl = xgb.XGBClassifier(
    objective='multi:softmax', num_class=10, random_state=42, tree_method='hist',
)
model_default_abl.fit(X_train_abl, y_train)
y_pred_default_abl = model_default_abl.predict(X_test_abl)
macro_f1_default_abl = f1_score(y_test, y_pred_default_abl, average='macro')

# Tuned XGBoost (no CZ_TYPE) — reuse best params from Week 2 grid search
model_tuned_abl = xgb.XGBClassifier(
    objective='multi:softmax', num_class=10, random_state=42, tree_method='hist',
    **best_params,
)
model_tuned_abl.fit(X_train_abl, y_train)
y_pred_tuned_abl = model_tuned_abl.predict(X_test_abl)
macro_f1_tuned_abl = f1_score(y_test, y_pred_tuned_abl, average='macro')

# TabNet (no CZ_TYPE) — rebuild cat indices/dims since CZ_TYPE_ENC is gone
X_train_abl_np = X_train_abl.values.astype(np.float32)
X_test_abl_np = X_test_abl.values.astype(np.float32)
cat_cols_abl = ['STATE_ENC', 'WFO_ENC']
cat_idxs_abl = [feature_cols_ablation.index(c) for c in cat_cols_abl]
cat_dims_abl = [int(df_imp[c].nunique()) for c in cat_cols_abl]

tabnet_abl = TabNetClassifier(
    cat_idxs=cat_idxs_abl, cat_dims=cat_dims_abl, cat_emb_dim=1,
    n_d=8, n_a=8, n_steps=3, gamma=1.3,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device, verbose=10,
)
tabnet_abl.fit(
    X_train_abl_np, y_train,
    eval_set=[(X_test_abl_np, y_test)],
    eval_metric=["accuracy"],
    max_epochs=100, patience=15, batch_size=1024,
)
y_pred_tabnet_abl = tabnet_abl.predict(X_test_abl_np)
macro_f1_tabnet_abl = f1_score(y_test, y_pred_tabnet_abl, average='macro')

print("=" * 70)
print("Ablation Study — CZ_TYPE_ENC removed")
print("=" * 70)
print(f"{'Model':<25} {'With CZ_TYPE':>14} {'Without':>10} {'Delta':>10}")
print("-" * 70)
print(f"{'Default XGBoost':<25} {macro_f1_default:>14.4f} {macro_f1_default_abl:>10.4f} {macro_f1_default_abl - macro_f1_default:>+10.4f}")
print(f"{'Tuned XGBoost':<25} {macro_f1_tuned:>14.4f} {macro_f1_tuned_abl:>10.4f} {macro_f1_tuned_abl - macro_f1_tuned:>+10.4f}")
print(f"{'TabNet':<25} {macro_f1_tabnet:>14.4f} {macro_f1_tabnet_abl:>10.4f} {macro_f1_tabnet_abl - macro_f1_tabnet:>+10.4f}")

print("\n" + "=" * 70)
print("Tuned XGBoost (no CZ_TYPE) — per-class report")
print("=" * 70)
print(classification_report(y_test, y_pred_tuned_abl, target_names=le_y.classes_))

# Feature importance without CZ_TYPE — check what the model relies on instead
imp_tuned_abl = pd.Series(model_tuned_abl.feature_importances_, index=feature_cols_ablation)
print("=" * 70)
print("Tuned XGBoost Feature Importance (no CZ_TYPE)")
print("=" * 70)
print(imp_tuned_abl.sort_values(ascending=False).to_string(float_format='%.4f'))

# TabNet Hyperparameter Tuning

Search over `n_d`/`n_a` width, `n_steps`, and learning rate. TabNet ties `n_d = n_a` by convention, so we sweep them together. Uses a single held-out validation split (carved from training) rather than full k-fold CV to keep runtime manageable.

In [ ]:
import itertools

# Carve a validation set from training so the test set stays untouched during tuning
X_tr_tn, X_val_tn, y_tr_tn, y_val_tn = train_test_split(
    X_train_np, y_train, stratify=y_train, test_size=0.2, random_state=42
)

tabnet_grid = {
    'n_da': [8, 16, 32],         # n_d == n_a
    'n_steps': [3, 5, 7],
    'lr': [1e-2, 2e-2, 5e-2],
}

results = []
configs = list(itertools.product(tabnet_grid['n_da'], tabnet_grid['n_steps'], tabnet_grid['lr']))
print(f"Running {len(configs)} TabNet configs...")

for i, (n_da, n_steps, lr) in enumerate(configs, 1):
    print(f"\n[{i}/{len(configs)}] n_d=n_a={n_da}, n_steps={n_steps}, lr={lr}")
    clf = TabNetClassifier(
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
        n_d=n_da, n_a=n_da, n_steps=n_steps, gamma=1.3,
        optimizer_params=dict(lr=lr),
        scheduler_params={"step_size": 50, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        device_name=device, verbose=0,
    )
    clf.fit(
        X_tr_tn, y_tr_tn,
        eval_set=[(X_val_tn, y_val_tn)],
        eval_metric=["accuracy"],
        max_epochs=100, patience=15, batch_size=1024,
    )
    val_pred = clf.predict(X_val_tn)
    val_f1 = f1_score(y_val_tn, val_pred, average='macro')
    results.append({'n_da': n_da, 'n_steps': n_steps, 'lr': lr, 'val_macro_f1': val_f1})
    print(f"  val macro F1: {val_f1:.4f}")

results_df = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
print("\n" + "=" * 70)
print("TabNet Tuning Results (sorted by val macro F1)")
print("=" * 70)
print(results_df.to_string(float_format='%.4f'))

best = results_df.iloc[0]
print(f"\nBest config: n_d=n_a={int(best['n_da'])}, n_steps={int(best['n_steps'])}, lr={best['lr']}")
print(f"Best val macro F1: {best['val_macro_f1']:.4f}")

# Retrain best config on full training set and evaluate on held-out test set
tabnet_best = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
    n_d=int(best['n_da']), n_a=int(best['n_da']), n_steps=int(best['n_steps']), gamma=1.3,
    optimizer_params=dict(lr=float(best['lr'])),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device, verbose=10,
)
tabnet_best.fit(
    X_train_np, y_train,
    eval_set=[(X_test_np, y_test)],
    eval_metric=["accuracy"],
    max_epochs=100, patience=15, batch_size=1024,
)
y_pred_tabnet_best = tabnet_best.predict(X_test_np)
macro_f1_tabnet_best = f1_score(y_test, y_pred_tabnet_best, average='macro')

print("\n" + "=" * 70)
print("Final Model Comparison (Macro F1 on test set)")
print("=" * 70)
print(f"  Default XGBoost:      {macro_f1_default:.4f}")
print(f"  Tuned XGBoost:        {macro_f1_tuned:.4f}")
print(f"  TabNet Baseline:      {macro_f1_tabnet:.4f}")
print(f"  TabNet Tuned:         {macro_f1_tabnet_best:.4f}")

print("\n" + classification_report(y_test, y_pred_tabnet_best, target_names=le_y.classes_))

# Per-Class ROC-AUC

Listed in the proposal as a primary evaluation metric. Computes one-vs-rest ROC-AUC per class for each model so we can see which event types each model separates cleanly versus where the boundaries blur.

In [ ]:
from sklearn.metrics import roc_auc_score

def per_class_auc(model, X, y, name):
    proba = model.predict_proba(X)
    auc_per = roc_auc_score(y, proba, multi_class='ovr', average=None)
    auc_macro = roc_auc_score(y, proba, multi_class='ovr', average='macro')
    return pd.Series(auc_per, index=le_y.classes_, name=name), float(auc_macro)

auc_def, m_auc_def   = per_class_auc(model_default, X_test, y_test, 'Default XGB')
auc_tun, m_auc_tun   = per_class_auc(model,         X_test, y_test, 'Tuned XGB')
auc_tnb, m_auc_tnb   = per_class_auc(tabnet,        X_test_np, y_test, 'TabNet Base')
auc_tnt, m_auc_tnt   = per_class_auc(tabnet_best,   X_test_np, y_test, 'TabNet Tuned')

auc_df = pd.concat([auc_def, auc_tun, auc_tnb, auc_tnt], axis=1)

print("=" * 70)
print("Macro ROC-AUC (one-vs-rest)")
print("=" * 70)
print(f"  Default XGBoost:  {m_auc_def:.4f}")
print(f"  Tuned XGBoost:    {m_auc_tun:.4f}")
print(f"  TabNet Baseline:  {m_auc_tnb:.4f}")
print(f"  TabNet Tuned:     {m_auc_tnt:.4f}")

print("\n" + "=" * 70)
print("Per-Class ROC-AUC")
print("=" * 70)
print(auc_df.to_string(float_format='%.4f'))

# Temporal Split Evaluation (Train 2023, Test 2024)

Verifies that the random-split numbers aren't artifacts of mixing years. Trains the tuned XGBoost and tuned TabNet on 2023 events only and evaluates on 2024 events. Imputation medians are re-fit on 2023 rows alone so the 2024 set is fully held out.

In [ ]:
# Re-derive year from BEGIN_DT (already parsed in df_imp)
years_arr = df_imp['BEGIN_DT'].dt.year.values
mask_2023 = years_arr == 2023
mask_2024 = years_arr == 2024

# Re-fit hierarchical lat/lon imputation on 2023 only
df_t = df.copy()
train_2023_coords = df_t[mask_2023 & df_t['BEGIN_LAT'].notna()]
med_cz_t    = train_2023_coords.groupby(['STATE', 'CZ_FIPS'])[['BEGIN_LAT', 'BEGIN_LON']].median()
med_wfo_t   = train_2023_coords.groupby(['STATE', 'WFO'])[['BEGIN_LAT', 'BEGIN_LON']].median()
med_state_t = train_2023_coords.groupby('STATE')[['BEGIN_LAT', 'BEGIN_LON']].median()

m = df_t['BEGIN_LAT'].isna()
for (s, cz), row in med_cz_t.iterrows():
    sel = m & (df_t['STATE'] == s) & (df_t['CZ_FIPS'] == cz)
    df_t.loc[sel, ['BEGIN_LAT', 'BEGIN_LON']] = row.values
m = df_t['BEGIN_LAT'].isna()
for (s, wfo), row in med_wfo_t.iterrows():
    sel = m & (df_t['STATE'] == s) & (df_t['WFO'] == wfo)
    df_t.loc[sel, ['BEGIN_LAT', 'BEGIN_LON']] = row.values
m = df_t['BEGIN_LAT'].isna()
for s, row in med_state_t.iterrows():
    sel = m & (df_t['STATE'] == s)
    df_t.loc[sel, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

X_train_t = df_t.loc[mask_2023, feature_cols]
X_test_t  = df_t.loc[mask_2024, feature_cols]
y_train_t = y_enc[mask_2023]
y_test_t  = y_enc[mask_2024]

print(f"Train (2023): {X_train_t.shape}, Test (2024): {X_test_t.shape}")

# Tuned XGBoost on temporal split
model_t_xgb = xgb.XGBClassifier(
    objective='multi:softmax', num_class=10, random_state=42, tree_method='hist',
    **best_params,
)
model_t_xgb.fit(X_train_t, y_train_t)
y_pred_t_xgb = model_t_xgb.predict(X_test_t)
macro_f1_t_xgb = f1_score(y_test_t, y_pred_t_xgb, average='macro')

# Tuned TabNet on temporal split (use best config from the sweep)
X_train_t_np = X_train_t.values.astype(np.float32)
X_test_t_np  = X_test_t.values.astype(np.float32)

tabnet_t = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
    n_d=int(best['n_da']), n_a=int(best['n_da']), n_steps=int(best['n_steps']), gamma=1.3,
    optimizer_params=dict(lr=float(best['lr'])),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device, verbose=10,
)
tabnet_t.fit(
    X_train_t_np, y_train_t,
    eval_set=[(X_test_t_np, y_test_t)],
    eval_metric=["accuracy"],
    max_epochs=100, patience=15, batch_size=1024,
)
y_pred_t_tn = tabnet_t.predict(X_test_t_np)
macro_f1_t_tn = f1_score(y_test_t, y_pred_t_tn, average='macro')

print("=" * 70)
print("Temporal Split vs Random Split (Macro F1)")
print("=" * 70)
print(f"{'Model':<22} {'Random':>10} {'2023->2024':>12} {'Delta':>10}")
print("-" * 70)
print(f"{'Tuned XGBoost':<22} {macro_f1_tuned:>10.4f} {macro_f1_t_xgb:>12.4f} {macro_f1_t_xgb - macro_f1_tuned:>+10.4f}")
print(f"{'Tuned TabNet':<22} {macro_f1_tabnet_best:>10.4f} {macro_f1_t_tn:>12.4f} {macro_f1_t_tn - macro_f1_tabnet_best:>+10.4f}")

print("\n" + "=" * 70)
print("Tuned XGBoost (2023->2024) per-class report")
print("=" * 70)
print(classification_report(y_test_t, y_pred_t_xgb, target_names=le_y.classes_))

# NaN-Coordinate Policy: Impute vs Drop

About 39.7% of records have missing `BEGIN_LAT` / `BEGIN_LON`. The current pipeline imputes via a hierarchical STATE/CZ_FIPS to STATE/WFO to STATE median fill. The alternative is to drop NaN-coord rows entirely.

To compare fairly, we train two variants of each tuned model (impute = full train set with imputed coords, drop = native-coord train rows only) and evaluate both on the **same** native-coord test subset. This isolates the training-data choice without conflating it with a different test distribution.

In [ ]:
# Native-coord masks (use original df, which still has the pre-imputation NaNs)
native_train_mask = df.loc[X_train.index, 'BEGIN_LAT'].notna().values
native_test_mask  = df.loc[X_test.index,  'BEGIN_LAT'].notna().values

# DROP variant: train + eval only on native-coord rows
X_train_drop = X_train.iloc[native_train_mask]
X_test_drop  = X_test.iloc[native_test_mask]
y_train_drop = y_train[native_train_mask]
y_test_drop  = y_test[native_test_mask]

print(f"Impute train: {len(X_train)} rows | Drop train: {len(X_train_drop)} ({len(X_train_drop)/len(X_train):.1%})")
print(f"Impute test:  {len(X_test)} rows  | Drop test:  {len(X_test_drop)} ({len(X_test_drop)/len(X_test):.1%})")

# Tuned XGBoost — DROP train
model_drop_xgb = xgb.XGBClassifier(
    objective='multi:softmax', num_class=10, random_state=42, tree_method='hist',
    **best_params,
)
model_drop_xgb.fit(X_train_drop, y_train_drop)
f1_drop_xgb = f1_score(y_test_drop, model_drop_xgb.predict(X_test_drop), average='macro')

# Tuned XGBoost — IMPUTE train, evaluated on the same drop test subset
f1_imp_xgb_on_drop = f1_score(y_test_drop, model.predict(X_test_drop), average='macro')

# Tuned TabNet — DROP train
X_train_drop_np = X_train_drop.values.astype(np.float32)
X_test_drop_np  = X_test_drop.values.astype(np.float32)

tabnet_drop = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
    n_d=int(best['n_da']), n_a=int(best['n_da']), n_steps=int(best['n_steps']), gamma=1.3,
    optimizer_params=dict(lr=float(best['lr'])),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device, verbose=10,
)
tabnet_drop.fit(
    X_train_drop_np, y_train_drop,
    eval_set=[(X_test_drop_np, y_test_drop)],
    eval_metric=["accuracy"],
    max_epochs=100, patience=15, batch_size=1024,
)
f1_drop_tn = f1_score(y_test_drop, tabnet_drop.predict(X_test_drop_np), average='macro')

# Tuned TabNet — IMPUTE train, evaluated on the same drop test subset
f1_imp_tn_on_drop = f1_score(y_test_drop, tabnet_best.predict(X_test_drop_np), average='macro')

print("\n" + "=" * 70)
print("Impute vs Drop (head-to-head on common native-coord test subset)")
print("=" * 70)
print(f"{'Model':<22} {'Impute':>10} {'Drop':>10} {'Delta':>10}")
print("-" * 70)
print(f"{'Tuned XGBoost':<22} {f1_imp_xgb_on_drop:>10.4f} {f1_drop_xgb:>10.4f} {f1_drop_xgb - f1_imp_xgb_on_drop:>+10.4f}")
print(f"{'Tuned TabNet':<22} {f1_imp_tn_on_drop:>10.4f} {f1_drop_tn:>10.4f} {f1_drop_tn - f1_imp_tn_on_drop:>+10.4f}")

print("\n" + "=" * 70)
print("Reference: full imputed test set")
print("=" * 70)
print(f"  Tuned XGBoost (impute, full test): {macro_f1_tuned:.4f}")
print(f"  Tuned TabNet  (impute, full test): {macro_f1_tabnet_best:.4f}")

# Results Summary (Week 4 Lock-in)

Persists every headline metric to `reports/week4_results.json` so the Week 4 report can cite concrete numbers without re-running the notebook.

In [ ]:
import json
from pathlib import Path

results = {
    'random_split': {
        'default_xgb_macro_f1':      float(macro_f1_default),
        'tuned_xgb_macro_f1':        float(macro_f1_tuned),
        'tabnet_baseline_macro_f1':  float(macro_f1_tabnet),
        'tabnet_tuned_macro_f1':     float(macro_f1_tabnet_best),
    },
    'ablation_no_cz_type': {
        'default_xgb_macro_f1':  float(macro_f1_default_abl),
        'tuned_xgb_macro_f1':    float(macro_f1_tuned_abl),
        'tabnet_macro_f1':       float(macro_f1_tabnet_abl),
        'delta_default_xgb':     float(macro_f1_default_abl - macro_f1_default),
        'delta_tuned_xgb':       float(macro_f1_tuned_abl - macro_f1_tuned),
        'delta_tabnet':          float(macro_f1_tabnet_abl - macro_f1_tabnet),
    },
    'tabnet_sweep_best': {
        'n_da':            int(best['n_da']),
        'n_steps':         int(best['n_steps']),
        'lr':              float(best['lr']),
        'val_macro_f1':    float(best['val_macro_f1']),
        'test_macro_f1':   float(macro_f1_tabnet_best),
    },
    'macro_roc_auc': {
        'default_xgb':      float(m_auc_def),
        'tuned_xgb':        float(m_auc_tun),
        'tabnet_baseline':  float(m_auc_tnb),
        'tabnet_tuned':     float(m_auc_tnt),
    },
    'nan_coord_policy': {
        'native_coord_train_pct':  float(native_train_mask.mean()),
        'native_coord_test_pct':   float(native_test_mask.mean()),
        'impute_xgb_on_drop_test': float(f1_imp_xgb_on_drop),
        'drop_xgb_on_drop_test':   float(f1_drop_xgb),
        'delta_xgb':               float(f1_drop_xgb - f1_imp_xgb_on_drop),
        'impute_tabnet_on_drop_test': float(f1_imp_tn_on_drop),
        'drop_tabnet_on_drop_test':   float(f1_drop_tn),
        'delta_tabnet':               float(f1_drop_tn - f1_imp_tn_on_drop),
    },
    'temporal_split_2023_to_2024': {
        'tuned_xgb_macro_f1':     float(macro_f1_t_xgb),
        'tabnet_tuned_macro_f1':  float(macro_f1_t_tn),
        'delta_tuned_xgb':        float(macro_f1_t_xgb - macro_f1_tuned),
        'delta_tabnet_tuned':     float(macro_f1_t_tn - macro_f1_tabnet_best),
    },
}

out = Path('../reports/week4_results.json')
out.write_text(json.dumps(results, indent=2))
print(f"Saved results to {out.resolve()}")
print(json.dumps(results, indent=2))